# AI Meeting Assistant — reference notebook

**Important:** this is a **read/edit reference copy** of `app.py`, not a runnable Streamlit app.
Streamlit apps use a script-execution model driven by `streamlit run app.py` (they redraw top-to-bottom
on every widget interaction). Jupyter's cell-by-cell model can't reproduce that, so `st.button`,
`st.file_uploader`, tabs, etc. won't work interactively here.

Use this notebook to read, edit, and test individual functions (LLM helpers, JSON parsing, etc.)
in isolation. To actually run the assistant, use `app.py` with:

```bash
pip install -r requirements.txt
streamlit run app.py
```

**Fix applied:** the deployment error (`TypeError: Metaclasses with custom tp_new are not supported`)
was caused by the host running **Python 3.14**, which breaks protobuf's compiled C extension.
`requirements.txt` now pins `protobuf<5.0.0`, and `runtime.txt` / `.python-version` pin the host to
**Python 3.12**. Also removed `"gemini-3.5-flash"` from the model list — that model name doesn't exist
and would have caused API errors if selected.

In [ ]:
# STEP 1: Load modules
import os
import json
import re
import time
import uuid
from datetime import datetime

from dotenv import load_dotenv
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

In [ ]:
# STEP 2: Config — set your API key here for notebook testing
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    genai.configure(api_key=GOOGLE_API_KEY)

model_choice = "gemini-2.0-flash"

HISTORY_FILE = "meeting_history.json"
AUDIO_DIR = "meeting_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

In [ ]:
# STEP 3: History storage helpers
def load_history():
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except (json.JSONDecodeError, OSError):
            return []
    return []


def save_history(history):
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2, ensure_ascii=False)


def add_meeting_record(record):
    history = load_history()
    history.insert(0, record)
    save_history(history)

In [ ]:
# STEP 4: LLM helpers
def get_llm():
    return ChatGoogleGenerativeAI(model=model_choice, temperature=0.3)


def transcribe_audio(file_path: str) -> str:
    """Transcribe audio using Gemini's native audio understanding."""
    audio_file = genai.upload_file(path=file_path)

    while audio_file.state.name == "PROCESSING":
        time.sleep(1)
        audio_file = genai.get_file(audio_file.name)

    if audio_file.state.name == "FAILED":
        raise RuntimeError("Audio processing failed on Google's servers.")

    model = genai.GenerativeModel(model_choice)
    response = model.generate_content(
        [
            "Transcribe this meeting recording as accurately as possible. "
            "Label speakers as Speaker 1, Speaker 2, etc. if they are distinguishable. "
            "Return only the transcript text.",
            audio_file,
        ]
    )
    return response.text


def extract_json_block(text: str):
    """Pull the first JSON array/object out of a model response."""
    candidates = []

    fenced = re.search(r"```(?:json)?\s*(\[.*?\]|\{.*?\})\s*```", text, re.DOTALL)
    if fenced:
        candidates.append(fenced.group(1))

    array_match = re.search(r"\[.*?\]", text, re.DOTALL)
    if array_match:
        candidates.append(array_match.group(0))

    obj_match = re.search(r"\{.*?\}", text, re.DOTALL)
    if obj_match:
        candidates.append(obj_match.group(0))

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue

    raise ValueError("No valid JSON found in model output.")

In [ ]:
def summarize_meeting(llm, transcript: str) -> str:
    prompt = ChatPromptTemplate.from_template(
        """You are an expert meeting assistant. Summarize the following meeting
transcript/notes into a concise, well-organized summary using short paragraphs
or bullet points. Cover: purpose of the meeting, key discussion points,
decisions made, and open questions. Do not invent details not present in the text.

Meeting content:
{content}

Summary:"""
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"content": transcript})

In [ ]:
def extract_action_items(llm, transcript: str, team_members: list) -> list:
    team_hint = (
        f"Known team members: {', '.join(team_members)}. Assign items to one of "
        f"these names ONLY if the transcript clearly indicates who owns the task; "
        f"otherwise use \"Unassigned\"."
        if team_members
        else 'No team member list was provided, so set "owner" to the name mentioned '
        'in the transcript, or "Unassigned" if unclear.'
    )

    prompt = ChatPromptTemplate.from_template(
        """Extract every action item / task from the following meeting content.
{team_hint}

Return ONLY valid JSON: a list of objects with keys "task", "owner", and "deadline"
("deadline" should be a date/timeframe if mentioned, otherwise "TBD"). No markdown
fences, no commentary — JSON only. If there are no action items, return [].

Meeting content:
{content}

JSON:"""
    )
    chain = prompt | llm | StrOutputParser()
    raw = chain.invoke({"content": transcript, "team_hint": team_hint})
    try:
        items = extract_json_block(raw)
        if isinstance(items, dict):
            items = [items]
        cleaned = []
        for item in items:
            if isinstance(item, dict):
                cleaned.append(item)
            else:
                cleaned.append({"task": str(item), "owner": "Unassigned", "deadline": "TBD"})
        return cleaned
    except (ValueError, json.JSONDecodeError):
        return [{"task": raw.strip(), "owner": "Unassigned", "deadline": "TBD"}]

In [ ]:
def draft_followup_email(llm, summary: str, action_items: list, title: str) -> str:
    items_text = "\n".join(
        f"- {i.get('task', '')} (Owner: {i.get('owner', 'Unassigned')}, Due: {i.get('deadline', 'TBD')})"
        for i in action_items
    ) or "- No specific action items were identified."

    prompt = ChatPromptTemplate.from_template(
        """Write a professional, concise follow-up email to meeting attendees.

Meeting title: {title}

Summary:
{summary}

Action items:
{items}

The email should include: a brief thank-you/recap line, the key summary points
as short bullets, a clearly formatted action items section (task, owner, due date),
and a friendly closing line. Include a subject line at the top formatted as
"Subject: ...". Return only the email text."""
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"title": title, "summary": summary, "items": items_text})

## Example usage (uncomment and run once GOOGLE_API_KEY is set)

In [ ]:
# transcript = "Paste a meeting transcript or notes here..."
# team_members = ["Alice", "Bob"]
#
# llm = get_llm()
# summary = summarize_meeting(llm, transcript)
# action_items = extract_action_items(llm, transcript, team_members)
# email_draft = draft_followup_email(llm, summary, action_items, "Weekly Sync")
#
# print(summary)
# print(action_items)
# print(email_draft)

---
## New features (mirrors `app.py`)

These four cells give you the **underlying logic** for the new features. The parts that need
live interactivity — the chat widget, checkboxes to mark items done, and the auto-updating
dashboard chart — only work inside the Streamlit app (`app.py`), since Jupyter can't run
Streamlit's widget/rerun loop. Run `streamlit run app.py` for the actual working demo.

1. **Ask Your Meeting** → `answer_meeting_question()`
2. **Automatic MoM (PDF + Markdown)** → `generate_mom_markdown()`, `generate_mom_pdf()`
3. **Multilingual output** → `LANGUAGE_INSTRUCTIONS`, passed into every prompt
4. **Team dashboard aggregation** → `build_dashboard_dataframe()`

⚠️ **PDF + Hindi note:** the PDF uses a Latin-only font. Devanagari (Hindi script) won't render
correctly in the PDF in this offline environment — use the Markdown MoM for Hindi content.

In [ ]:
# Multilingual support — language instructions injected into every prompt
LANGUAGE_INSTRUCTIONS = {
    "English": "Respond in clear, professional English.",
    "Hindi": "Respond in Hindi, written in Devanagari script.",
    "Hinglish": (
        "Respond in Hinglish — a natural mix of Hindi and English written in "
        "Roman/Latin script, the way it's commonly written in Indian workplace "
        "chats and emails."
    ),
}

# Example: pick a language, then pass LANGUAGE_INSTRUCTIONS[language] into
# summarize_meeting(llm, transcript, language_instruction) etc. (see previous cells).

In [ ]:
# Feature 1: Ask Your Meeting — question answering grounded in one meeting's transcript
def answer_meeting_question(llm, transcript: str, chat_history: list, question: str, language_instruction: str) -> str:
    history_text = "\n".join(f"{role}: {msg}" for role, msg in chat_history[-6:]) or "(no prior turns)"
    prompt = ChatPromptTemplate.from_template(
        """You are answering questions about ONE specific meeting, using ONLY the
meeting content below as your source of truth. If the answer isn't in the
content, say plainly that it wasn't mentioned in the meeting — never invent
details. {language_instruction}

Meeting content:
{content}

Conversation so far:
{history}

Question: {question}

Answer:"""
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({
        "content": transcript, "history": history_text,
        "question": question, "language_instruction": language_instruction,
    })

# Example:
# llm = get_llm()
# chat_history = []
# answer = answer_meeting_question(llm, transcript, chat_history, "What did Rahul agree to do?", LANGUAGE_INSTRUCTIONS["English"])
# print(answer)

In [ ]:
# Feature 2: Automatic Meeting Minutes (Markdown + PDF)
from fpdf import FPDF

def generate_mom_markdown(meeting_title: str, meeting_date: str, summary: str, action_items: list) -> str:
    """Markdown MoM — safe for any language/script, including Hindi."""
    lines = [f"# Minutes of Meeting: {meeting_title}", "", f"**Date:** {meeting_date}", "", "## Summary", "", summary, "", "## Action Items", ""]
    if action_items:
        for i, item in enumerate(action_items, 1):
            lines.append(
                f"{i}. **{item.get('task', '')}** — Owner: {item.get('owner', 'Unassigned')}, "
                f"Due: {item.get('deadline', 'TBD')}, Status: {item.get('status', 'Pending')}"
            )
    else:
        lines.append("No action items identified.")
    return "\n".join(lines)


def generate_mom_pdf(meeting_title: str, meeting_date: str, summary: str, action_items: list) -> bytes:
    """PDF MoM. Latin-only font: best for English/Hinglish. Not for Devanagari Hindi text."""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    pdf.set_font("Helvetica", "B", 16)
    pdf.multi_cell(0, 10, "Minutes of Meeting", align="C")
    pdf.ln(2)

    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 7, f"Meeting: {meeting_title}")
    pdf.multi_cell(0, 7, f"Date: {meeting_date}")
    pdf.ln(4)

    pdf.set_font("Helvetica", "B", 13)
    pdf.multi_cell(0, 8, "Summary")
    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 6, summary.encode("latin-1", "replace").decode("latin-1"))
    pdf.ln(4)

    pdf.set_font("Helvetica", "B", 13)
    pdf.multi_cell(0, 8, "Action Items")
    pdf.set_font("Helvetica", "", 11)
    if action_items:
        for i, item in enumerate(action_items, 1):
            line = (
                f"{i}. {item.get('task', '')} | Owner: {item.get('owner', 'Unassigned')} | "
                f"Due: {item.get('deadline', 'TBD')} | Status: {item.get('status', 'Pending')}"
            )
            pdf.multi_cell(0, 6, line.encode("latin-1", "replace").decode("latin-1"))
    else:
        pdf.multi_cell(0, 6, "No action items identified.")

    return bytes(pdf.output(dest="S"))

# Example:
# md_text = generate_mom_markdown("Weekly Sync", "2026-08-07", summary, action_items)
# pdf_bytes = generate_mom_pdf("Weekly Sync", "2026-08-07", summary, action_items)
# with open("weekly_sync_mom.pdf", "wb") as f:
#     f.write(pdf_bytes)

In [ ]:
# Feature 4: Team Performance Dashboard — aggregation logic
import pandas as pd

def build_dashboard_dataframe(history: list) -> pd.DataFrame:
    """Flatten every action item across all saved meetings for the dashboard."""
    rows = []
    for record in history:
        for item in record.get("action_items", []):
            rows.append({
                "Meeting": record.get("title", ""),
                "Date": record.get("created_at", "")[:10],
                "Owner": item.get("owner", "Unassigned") or "Unassigned",
                "Task": item.get("task", ""),
                "Deadline": item.get("deadline", "TBD"),
                "Status": item.get("status", "Pending"),
            })
    return pd.DataFrame(rows, columns=["Meeting", "Date", "Owner", "Task", "Deadline", "Status"])

# Example (in the Streamlit app, this feeds st.bar_chart for a live dashboard):
# history = load_history()
# df = build_dashboard_dataframe(history)
# summary_by_owner = df.groupby(["Owner", "Status"]).size().unstack(fill_value=0)
# print(summary_by_owner)